# 1. Setting up the Environment

---

## 1.1 Installing the Packages

In [2]:
!pip install GPUtil

  Preparing metadata (setup.py) ... done
  Created wheel for GPUtil: filename=GPUtil-1.4.0-py3-none-any.whl size=7392 sha256=8991cfbdb641bb99afbe6ef78df1c76d1a0719aecef6dfb573c6da23389b3990
  Stored in directory: /root/.cache/pip/wheels/a9/8a/bd/81082387151853ab8b6b3ef33426e98f5cbfebc3c397a9d4d0
Successfully built GPUtil


## 1.2 Importing the required Libraries

In [3]:
import re
import sqlite3
import datetime
import json


import torch
import GPUtil

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

## 1.3 Getting the GPU ready

In [4]:
print("GPUs available:")
GPUtil.showUtilization()

MODEL_NAME = "microsoft/phi-2"
device = 0 if torch.cuda.is_available() else -1
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.to("cuda" if device == 0 else "cpu")

text_generator = pipeline("text-generation", model=model, tokenizer=tokenizer, device=device)

print("Loaded model:", MODEL_NAME)

GPUs available:
| ID | GPU | MEM |
------------------
|  0 |  0% |  0% |


tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0


Loaded model: microsoft/phi-2


# 2. Setting up the Database

---

In [5]:
conn = sqlite3.connect("expenses.db")
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS expenses (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    description TEXT,
    amount INTEGER,
    date TEXT,
    categories TEXT
)
''')
conn.commit()

print("Database initialized.")

Database initialized.


# 3. Helper functions for extracting the Information and processing the Query

---

In [6]:
def parse_date(message: str) -> datetime.date:
    today = datetime.date.today()
    if "yesterday" in message.lower():
        return today - datetime.timedelta(days=1)
    return today

def categorize_expense(text: str) -> list:
    text_lower = text.lower()
    categories = set()
    if any(kw in text_lower for kw in ["coffee", "latte", "cappuccino", "espresso", "filter coffee", "cold coffee"]):
        categories.add("coffee")
    if any(kw in text_lower for kw in ["dinner", "biryani", "snacks", "sandwich", "food", "cafe", "café", "cutting chai"]):
        categories.add("food")
    if any(kw in text_lower for kw in ["ola", "uber", "train", "cab", "taxi"]):
        categories.add("travel")
    if any(kw in text_lower for kw in ["groceries", "zepto"]):
        categories.add("groceries")
    if any(kw in text_lower for kw in ["jeans", "shopping", "levis", "levi's"]):
        categories.add("shopping")
    return list(categories)

def extract_expense_details(message: str) -> dict:
    match = re.search(r"(?:₹|Rs\.?|INR)\s?(\d+)", message, flags=re.IGNORECASE)
    if not match:
        raise ValueError("No amount found in the message.")
    amount = int(match.group(1))
    date = parse_date(message)
    description = message.strip()
    cats = categorize_expense(description)
    categories = json.dumps(cats)
    return {"amount": amount, "description": description, "date": date.isoformat(), "categories": categories}

def classify_message_llm(message: str) -> str:
    prompt = (
        "Decide whether the following message is an expense entry or a query about expenses. "
        "If it is adding an expense, answer 'expense'; if it is asking about expenses, answer 'query'.\n"
        f"Message: \"{message}\"\nAnswer:"
    )
    result = text_generator(prompt, max_length=len(prompt)+10, num_return_sequences=1)
    generated_text = result[0]['generated_text'][len(prompt):].strip().lower()
    if "expense" in generated_text:
        return "expense"
    elif "query" in generated_text or "how" in generated_text or "what" in generated_text:
        return "query"
    else:
        return "expense"

def classify_message(message: str) -> str:
    message_lower = message.lower().strip()
    if "₹" in message_lower or re.search(r"\brs\.?\s?\d+", message_lower):
        return "expense"
    if message_lower.endswith("?") or any(message_lower.startswith(kw) for kw in ["how", "what", "show", "list", "total", "biggest"]):
        return "query"
    return classify_message_llm(message)

# 4. The ExpenseTracker

---

In [7]:
class ExpenseTracker:
    def __init__(self, db_connection):
        self.conn = db_connection
        self.cursor = self.conn.cursor()

    def add_expense(self, message: str) -> str:
        try:
            details = extract_expense_details(message)
            self.cursor.execute(
                '''
                INSERT INTO expenses (description, amount, date, categories)
                VALUES (?, ?, ?, ?)
                ''',
                (
                    details["description"],
                    details["amount"],
                    details["date"],
                    details["categories"],
                ),
            )
            self.conn.commit()
            return (
                f"Expense added: ₹{details['amount']} on {details['date']}.\n"
                f"Details: {details['description']} | Categories: {json.loads(details['categories'])}"
            )
        except Exception as e:
            return f"Failed to add expense: {e}"

    def process_query(self, query: str) -> str:
        query_lower = query.lower()
        response_lines = []

        def fetch_rows(sql_query, params=()):
            self.cursor.execute(sql_query, params)
            return self.cursor.fetchall()

        if ("spent so far" in query_lower) or ("total expense" in query_lower and "on" not in query_lower):
            self.cursor.execute("SELECT SUM(amount) FROM expenses")
            total = self.cursor.fetchone()[0] or 0
            response_lines.append(f"Your total expenses so far are: ₹{total}")
            response_lines.append("Detailed Expenses:")
            rows = fetch_rows("SELECT description, amount, date, categories FROM expenses ORDER BY date ASC")
            for row in rows:
                description, amount, date, categories = row
                try:
                    cats = json.loads(categories)
                except:
                    cats = categories
                response_lines.append(
                    f"  • {description} | Amount: ₹{amount} | Date: {date} | Categories: "
                    f"{', '.join(cats) if isinstance(cats, list) else cats}"
                )

        elif ("total expense" in query_lower and "on" in query_lower) or any(
            kw in query_lower for kw in ["coffee", "food", "groceries", "travel", "online food"]
        ):
            if "online food" in query_lower:
                rows = fetch_rows(
                    "SELECT description, amount, date, categories FROM expenses WHERE lower(description) LIKE ? OR lower(description) LIKE ?",
                    ("%%swiggy%%", "%%blinkit%%"),
                )
                total = sum(row[1] for row in rows)
                response_lines.append(f"Your total spending on online food orders is: ₹{total}")
                response_lines.append("Detailed Online Food Orders:")
                for row in rows:
                    description, amount, date, categories = row
                    try:
                        cats = json.loads(categories)
                    except:
                        cats = categories
                    response_lines.append(
                        f"  • {description} | Amount: ₹{amount} | Date: {date} | Categories: "
                        f"{', '.join(cats) if isinstance(cats, list) else cats}"
                    )
            else:
                category = None
                for cat in ["coffee", "food", "groceries", "travel"]:
                    if cat in query_lower:
                        category = cat
                        break
                if category:
                    rows = fetch_rows(
                        "SELECT description, amount, date, categories FROM expenses WHERE lower(categories) LIKE ?",
                        (f"%{category}%",),
                    )
                    total = sum(row[1] for row in rows)
                    response_lines.append(f"Your total {category} expenditure is: ₹{total}")
                    response_lines.append(f"Detailed {category.capitalize()} Expenses:")
                    for row in rows:
                        description, amount, date, categories = row
                        try:
                            cats = json.loads(categories)
                        except:
                            cats = categories
                        response_lines.append(
                            f"  • {description} | Amount: ₹{amount} | Date: {date} | Categories: "
                            f"{', '.join(cats) if isinstance(cats, list) else cats}"
                        )
                else:
                    response_lines.append("Could not determine the expense category from your query.")

        elif "yesterday" in query_lower:
            target_date = (datetime.date.today() - datetime.timedelta(days=1)).isoformat()
            rows = fetch_rows("SELECT description, amount, date, categories FROM expenses WHERE date = ?", (target_date,))
            total = sum(row[1] for row in rows)
            response_lines.append(f"Expenses from yesterday ({target_date}):")
            for row in rows:
                description, amount, date, categories = row
                try:
                    cats = json.loads(categories)
                except:
                    cats = categories
                response_lines.append(
                    f"  • {description} | Amount: ₹{amount} | Categories: "
                    f"{', '.join(cats) if isinstance(cats, list) else cats}"
                )
            response_lines.append(f"Total: ₹{total}")

        elif "biggest expense" in query_lower or "biggest expenses" in query_lower:
            if "this week" in query_lower:
                today = datetime.date.today()
                start_of_week = today - datetime.timedelta(days=today.weekday())
                end_of_week = start_of_week + datetime.timedelta(days=6)
                rows = fetch_rows(
                    "SELECT description, amount, date, categories FROM expenses WHERE date BETWEEN ? AND ? ORDER BY amount DESC",
                    (start_of_week.isoformat(), end_of_week.isoformat()),
                )
                response_lines.append("Your biggest expenses this week are:")
            else:
                rows = fetch_rows("SELECT description, amount, date, categories FROM expenses ORDER BY amount DESC")
                response_lines.append("Your biggest expenses so far are:")
            for row in rows[:5]:
                description, amount, date, categories = row
                try:
                    cats = json.loads(categories)
                except:
                    cats = categories
                response_lines.append(
                    f"  • {description} | Amount: ₹{amount} | Date: {date} | Categories: "
                    f"{', '.join(cats) if isinstance(cats, list) else cats}"
                )

        else:
            response_lines.append("I'm sorry, I couldn't understand your query. Please try rephrasing.")

        return "\n".join(response_lines)

    def process_message(self, message: str) -> str:
        msg_type = classify_message(message)
        if msg_type == "expense":
            return self.add_expense(message)
        elif msg_type == "query":
            return self.process_query(message)
        else:
            return "I'm not sure how to process your message."


tracker = ExpenseTracker(conn)
print("Expense Tracker Bot is ready.")

Expense Tracker Bot is ready.


# 5. Simulating the experience

---

In [8]:
cursor.execute("DELETE FROM expenses")
conn.commit()
print("Database cleared.")

expense_messages = [
    "Had a filter coffee at a local café, cost ₹50.",
    "Spent ₹400 on a cappuccino at Starbucks.",
    "Bought a cold coffee from CCD for ₹180.", 
    "₹120 for a cutting chai at a roadside stall.",
    "Dinner at a fine dining restaurant, cost ₹1800.", 
    "Took an Ola to work, cost ₹250.", 
    "Bought snacks from Blinkit for ₹300 yesterday.", 
    "Ordered biryani from Swiggy for ₹500.", 
    "Groceries from Zepto, cost ₹1000.", 
    "Paid ₹1500 for a new pair of jeans from Levi's.", 
    "Took an Uber to the airport, cost ₹600.", 
    "Booked a train ticket for my trip, cost ₹1200."
]

print("=== Adding Expenses ===")
for msg in expense_messages:
    response = tracker.process_message(msg)
    print(response)


queries = [
    "How much have I spent on coffee?",
    "What is my total food expenditure?",
    "What is my total expense so far?",
    "How much did I spend on groceries?",
    "How much did I spend on online food ordering?",
    "What are my expenses from yesterday?",
    "What is my total expense on travel?"
]

print("\n=== Processing Queries ===")
for q in queries:
    print("\nUser Query:", q)
    response = tracker.process_message(q)
    print("Bot Response:")
    print(response)

Database cleared.
=== Adding Expenses ===
Expense added: ₹50 on 2025-02-14.
Details: Had a filter coffee at a local café, cost ₹50. | Categories: ['coffee', 'food']
Expense added: ₹400 on 2025-02-14.
Details: Spent ₹400 on a cappuccino at Starbucks. | Categories: ['coffee']
Expense added: ₹180 on 2025-02-14.
Details: Bought a cold coffee from CCD for ₹180. | Categories: ['coffee']
Expense added: ₹120 on 2025-02-14.
Details: ₹120 for a cutting chai at a roadside stall. | Categories: ['food']
Expense added: ₹1800 on 2025-02-14.
Details: Dinner at a fine dining restaurant, cost ₹1800. | Categories: ['food']
Expense added: ₹250 on 2025-02-14.
Details: Took an Ola to work, cost ₹250. | Categories: ['travel']
Expense added: ₹300 on 2025-02-13.
Details: Bought snacks from Blinkit for ₹300 yesterday. | Categories: ['food']
Expense added: ₹500 on 2025-02-14.
Details: Ordered biryani from Swiggy for ₹500. | Categories: ['food']
Expense added: ₹1000 on 2025-02-14.
Details: Groceries from Zepto, c